In [1]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, current_timestamp, initcap, lit, to_timestamp, trim, when
)
from pyspark.sql.types import DecimalType
from pyspark.sql import Window
import pyspark.sql.functions as F

HIST_TABLE    = "raw_historical_transactions"
LIVE_TABLE    = "raw_live_transactions"
ACCTS_TABLE   = "silver_accounts"
TARGET        = "silver_transactions"
QUARANTINE    = "quarantine_transactions"
WM_TABLE      = "_pipeline_watermarks"
LAYER_KEY     = "silver_transactions"
TS_FORMAT     = "yyyy-MM-dd HH:mm:ss"

print("Config loaded.")

StatementMeta(, 49ac4009-d061-4595-8fcb-cac8e12c8976, 3, Finished, Available, Finished, False)

Config loaded.


In [3]:
last_watermark = (
    spark.table(WM_TABLE)
         .filter(f"layer_name = '{LAYER_KEY}'")
         .select("watermark_ts").collect()[0]["watermark_ts"]
)
print(f"Watermark: {last_watermark}")

hist_cols = spark.table(HIST_TABLE).columns
live_cols = spark.table(LIVE_TABLE).columns

# Historical: our PySpark notebook added ingestion_timestamp
if "ingestion_timestamp" in hist_cols:
    df_hist = spark.table(HIST_TABLE).filter(col("ingestion_timestamp") > last_watermark)
else:
    df_hist = spark.table(HIST_TABLE)   # first run — no column yet, take all rows

# Live: Eventstream uses EventProcessedUtcTime (its own ingestion stamp)
if "EventProcessedUtcTime" in live_cols:
    df_live = spark.table(LIVE_TABLE).filter(col("EventProcessedUtcTime") > last_watermark)
elif "ingestion_timestamp" in live_cols:
    df_live = spark.table(LIVE_TABLE).filter(col("ingestion_timestamp") > last_watermark)
else:
    df_live = spark.table(LIVE_TABLE)

hist_new = df_hist.count()
live_new  = df_live.count()
print(f"New historical rows: {hist_new}  |  New live rows: {live_new}")

if hist_new + live_new == 0:
    print("No new rows. Exiting.")
    spark.stop()
    notebookutils.notebook.exit("NO_NEW_DATA")

StatementMeta(, 629fb55b-305f-4e73-ae4f-f50caeb87048, 5, Finished, Available, Finished, False)

Watermark: 2000-01-01 00:00:00
New historical rows: 310  |  New live rows: 600


In [4]:
def normalise(df, src_name):
    # Add source_system if missing
    if "source_system" not in df.columns:
        df = df.withColumn("source_system", lit(src_name))

    # Standardise ingestion timestamp — each source calls it something different
    if "ingestion_timestamp" not in df.columns:
        if "EventProcessedUtcTime" in df.columns:
            df = df.withColumn("ingestion_timestamp", col("EventProcessedUtcTime"))
        else:
            df = df.withColumn("ingestion_timestamp", current_timestamp())

    # Drop Eventstream metadata columns — they don't belong in Silver
    eventstream_cols = ["EventProcessedUtcTime", "PartitionId", "EventEnqueuedUtcTime"]
    cols_to_drop = [c for c in eventstream_cols if c in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)

    return df.select(
        "transaction_id", "account_id", "amount", "merchant",
        "city", "transaction_timestamp", "ingestion_timestamp", "source_system"
    )

df_combined = normalise(df_hist, "ADLS_HISTORICAL").unionByName(
    normalise(df_live, "EVENTHUB_LIVE")
)

df_typed = (
    df_combined
    .withColumn("amount",               col("amount").cast(DecimalType(18,2)))
    .withColumn("transaction_timestamp",to_timestamp(col("transaction_timestamp"), TS_FORMAT))
    .withColumn("city",                 initcap(trim(col("city"))))
    .withColumn("merchant",             initcap(trim(col("merchant"))))
)
print(f"Combined rows: {df_typed.count()}")

StatementMeta(, 629fb55b-305f-4e73-ae4f-f50caeb87048, 6, Finished, Available, Finished, False)

Combined rows: 910


In [5]:
# Quarantine bad rows
cond_bad = (
    col("transaction_id").isNull() | col("account_id").isNull() |
    col("amount").isNull()         | (col("amount") < 0)
)
df_good = df_typed.filter(~cond_bad)
df_bad  = df_typed.filter(cond_bad)
print(f"Good: {df_good.count()}  |  Bad: {df_bad.count()}")

if df_bad.count() > 0:
    df_quarantine = (
        df_bad
        .withColumn("quarantine_reason",
            when(col("transaction_id").isNull(), lit("NULL_TRANSACTION_ID"))
            .when(col("account_id").isNull(),    lit("NULL_ACCOUNT_ID"))
            .when(col("amount").isNull(),         lit("NULL_AMOUNT"))
            .otherwise(                           lit("NEGATIVE_AMOUNT")))
        .withColumn("raw_json", F.to_json(F.struct([col(c) for c in df_bad.columns])))
        .withColumn("quarantined_at", current_timestamp())
        .select("transaction_id","account_id","amount",
                "raw_json","quarantine_reason","source_system","quarantined_at")
    )
    df_quarantine.write.mode("append").format("delta").saveAsTable(QUARANTINE)
    print(f"Quarantined: {df_bad.count()} rows")

StatementMeta(, 629fb55b-305f-4e73-ae4f-f50caeb87048, 7, Finished, Available, Finished, False)

Good: 910  |  Bad: 0


In [6]:
# Dedup within this batch
window_dedup = Window.partitionBy("transaction_id").orderBy(col("ingestion_timestamp").desc())
df_deduped = (
    df_good
    .withColumn("rn", F.row_number().over(window_dedup))
    .filter(col("rn") == 1).drop("rn")
)
print(f"After dedup: {df_deduped.count()}")

StatementMeta(, 629fb55b-305f-4e73-ae4f-f50caeb87048, 8, Finished, Available, Finished, False)

After dedup: 755


In [7]:
# Enrich with account dimensions
valid_ids    = spark.table(ACCTS_TABLE).select("account_id").distinct()
df_acct_slim = spark.table(ACCTS_TABLE).select(
    "account_id","home_city","risk_category","account_status","kyc_status"
)

df_enriched = (
    df_deduped
    .join(valid_ids, on="account_id", how="inner")
    .join(df_acct_slim, on="account_id", how="left")
    .withColumn("is_city_mismatch",
        when(col("city") != col("home_city"), lit(True)).otherwise(lit(False)))
    .withColumn("silver_updated_at", current_timestamp())
    .select(
        "transaction_id","account_id","amount","merchant","city",
        "transaction_timestamp","ingestion_timestamp","source_system",
        "home_city","risk_category","account_status","kyc_status",
        "is_city_mismatch","silver_updated_at"
    )
)
new_row_count = df_enriched.count()
print(f"Enriched rows ready to MERGE: {new_row_count}")

StatementMeta(, 629fb55b-305f-4e73-ae4f-f50caeb87048, 9, Finished, Available, Finished, False)

Enriched rows ready to MERGE: 755


In [8]:
if spark.catalog.tableExists(TARGET):
    dt = DeltaTable.forName(spark, TARGET)
    (dt.alias("t").merge(df_enriched.alias("s"), "t.transaction_id = s.transaction_id")
       .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    print(f"MERGE complete → {TARGET}")
else:
    df_enriched.write.mode("overwrite").format("delta").saveAsTable(TARGET)
    print(f"First run — created {TARGET}")

StatementMeta(, 629fb55b-305f-4e73-ae4f-f50caeb87048, 10, Finished, Available, Finished, False)

MERGE complete → silver_transactions


In [9]:
max_ts = df_combined.agg(F.max("ingestion_timestamp")).collect()[0][0]

spark.sql(f"""
    UPDATE {WM_TABLE}
    SET watermark_ts    = '{max_ts}',
        rows_last_run   = {new_row_count},
        last_run_status = 'SUCCESS',
        last_run_at     = current_timestamp()
    WHERE layer_name = '{LAYER_KEY}'
""")
print(f"Watermark updated to: {max_ts}")
spark.stop()

StatementMeta(, 629fb55b-305f-4e73-ae4f-f50caeb87048, 11, Finished, Available, Finished, False)

Watermark updated to: 2026-06-18 10:38:32.493790
